In [ ]:
from pathlib import Path
import cv2, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from tqdm import tqdm
import time


In [ ]:
# Image dataset preprocessing part
# I did the adjustment of contrast ratio, noise smoothing and rotation
def contrastRatio(img, clipLimit=2.5, tileGridSize=(10,10)):
    #We gonna use CLAHE (contrast limited adaptive histogram equalization) to do contrast
    #convert to lab space
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    #sparate different channels
    l, a, b = cv2.split(lab)
    #clahe constructor
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    #Light channel enhanced by contrast ratio
    cl = clahe.apply(l)
    #merge back to lab
    merged = cv2.merge((cl, a, b))
    #convert lab to BGR
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
def smoothBilateral(img, d=9, sigmaColor=75, sigmaSpace=75):
    #Bilaterao filter
    return cv2.bilateralFilter(img, d, sigmaColor, sigmaSpace)
def rotation(img):
    #right rotation
    return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

def resizing(img, size=(80,80)):
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)
def gammaAdjust(img, gamma=1.3):
    inv = 1.0 / gamma
    table = np.array([((i/255.0)**inv)*255 for i in np.arange(256)]).astype("uint8")
    return cv2.LUT(img, table)

def basic_preprocess(img, size=(80,80), gamma=1.3):
    img = resizing(img, size)           
    img = gammaAdjust(img, gamma)       
    return img
def main():
    try:
        base = Path(__file__).resolve().parent
    except NameError:
        base = Path().resolve()

    input_root  = base / "defungi"
    output_root = base / "defungiPreprocessed"
    subdirs     = ["H1", "H2", "H3", "H5", "H6"]

    for sub in subdirs:
        in_dir  = input_root  / sub
        out_dir = output_root / sub
        out_dir.mkdir(parents=True, exist_ok=True)

        imgs = [p for p in in_dir.iterdir() if p.suffix.lower() == ".jpg"]
        for img_path in tqdm(imgs, desc=f"Processing {sub}", unit="img"):
            data = img_path.read_bytes()
            img  = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                tqdm.write(f"Can't read {img_path.name}")
                continue
            
            basic = basic_preprocess(img)
            save_path = out_dir / img_path.name
            cv2.imencode(img_path.suffix, basic)[1].tofile(str(save_path))

        print(f"{sub} baseline‐processed and saved in {out_dir}")

if __name__ == "__main__":
    main()
#for train dataset
def train_augment(img):
    img = contrastRatio(img)
    img = smoothBilateral(img)
    return img
#differentiate the color based on color distribution
def extract_color_histogram(img, bins=(32,32,32)):
    histB = cv2.calcHist([img],[0],None,[bins[0]],[0,256]).flatten()
    histG = cv2.calcHist([img],[1],None,[bins[1]],[0,256]).flatten()
    histR = cv2.calcHist([img],[2],None,[bins[2]],[0,256]).flatten()
    hist = np.concatenate([histB, histG, histR])
    return hist / hist.sum()

# below is translating color to gray, catching the shape and texture of the images
hog_descriptor = cv2.HOGDescriptor(
    _winSize     = (80,80),
    _blockSize   = (16,16),
    _blockStride = (8,8),
    _cellSize    = (8,8),
    _nbins       = 9
)
def extract_hog_features_opencv(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    desc = hog_descriptor.compute(gray)
    return desc.flatten()


In [ ]:
base = Path().resolve() / "defungiPreprocessed"
subdirs = ["H1","H2","H3","H5","H6"]
label_map = {sub:i for i, sub in enumerate(subdirs)}
#1.go throught the preprocessed dataset
images_all, y_all = [], []
for sub in subdirs:
    for img_path in (base/sub).glob("*.jpg"):
        data = img_path.read_bytes()
        img  = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        if img is None: 
            continue
        images_all.append(img)
        y_all.append(label_map[sub])
y_all = np.array(y_all)
# 2) select 60% data for each dataset, therefore we have roughly 5400 pictures in the set
np.random.seed(1)
images, y = [], []
for cls in np.unique(y_all):
    idxs  = np.where(y_all == cls)[0]
    n_sel = int(len(idxs) * 1)
    chosen = np.random.choice(idxs, size=n_sel, replace=False)
    for i in chosen:
        images.append(images_all[i])
        y.append(y_all[i])
y = np.array(y)

#outer and inner level cv,outer has 5 folds, inner has 3 folds
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
#3 algorithms model
pipelines = {
    "NB": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    GaussianNB())
    ]),
    "LinearSVC": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LinearSVC(dual=True, max_iter=100000, tol=1e-3))
    ]),
    "DT": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    DecisionTreeClassifier())
    ])
}
#set up the hyperparameters for each models
param_grids = {
    "NB": {
        "clf__var_smoothing":[1e-10, 1e-8,1e-6] #
    },
    "LinearSVC": {
        "clf__C":[0.001,0.01, 0.1]
    },
    "DT": {
        "clf__max_depth": [None, 10, 20],          
        "clf__min_samples_split":[2, 5, 10],
        "clf__max_features":["sqrt"]

    }
}
accuracy_scores = {}
f1_scores       = {}
training_accuracy_scores = {}  
best_params_all = {}
runtime_scores = {}

for name, pipe in pipelines.items():
    acc_list    = []
    f1_list     = []
    train_acc_list = []           
    y_true_all  = []
    y_pred_all  = []
    best_params_all[name] = []
    runtime_scores[name] = []  

    print(f"=== {name} ===")
    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(images, y), start=1):
        # train set preprocessed (usually much more based on test set)
        X_train, y_train = [], []
        for i in train_idx:
            img = train_augment(images[i])
            img = resizing(img, size=(80,80))
            img = gammaAdjust(img, gamma=1.3)
            hist = extract_color_histogram(img, bins=(32,32,32))
            hogf = extract_hog_features_opencv(img)
            X_train.append(np.concatenate([hist, hogf]))
            y_train.append(y[i])
        X_train, y_train = np.vstack(X_train), np.array(y_train)

        # test set preprocessed
        X_test, y_test = [], []
        for i in test_idx:
            img = resizing(images[i], size=(80,80))
            img = gammaAdjust(img, gamma=1.3)
            hist = extract_color_histogram(img, bins=(32,32,32))
            hogf = extract_hog_features_opencv(img)
            X_test.append(np.concatenate([hist, hogf]))
            y_test.append(y[i])
        X_test, y_test = np.vstack(X_test), np.array(y_test)

        start = time.time() 
        # paramter in inner level
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grids[name],
            cv=inner_cv,
            scoring="accuracy",
            n_jobs=1,
            verbose=0
        )
        gs.fit(X_train, y_train)
        y_pred = gs.predict(X_test)
        elapsed = time.time() - start  # time calculation
        runtime_scores[name].append(elapsed)  # save time

        # use the best parameter to run CV and record CV results
        print(f" Fold {fold_idx} best params: {gs.best_params_}, CV acc: {gs.best_score_:.3f}")
        best_params_all[name].append(tuple(sorted(gs.best_params_.items())))

        #train set prediction and calculation
        y_train_pred = gs.predict(X_train)
        train_acc    = accuracy_score(y_train, y_train_pred)
        train_acc_list.append(train_acc)

        # test set prediction and calculation
        y_pred = gs.predict(X_test)
        acc    = accuracy_score(y_test, y_pred)
        f1     = f1_score(y_test, y_pred, average="macro")

        acc_list.append(acc)
        f1_list.append(f1)
        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

        print(f"          train acc = {train_acc:.3f}, test acc = {acc:.3f}, test f1 = {f1:.3f}")

    # most common parameters (usually the best in my case here)
    mc, count = Counter(best_params_all[name]).most_common(1)[0]
    print(f" {name} most freq best params: {dict(mc)} (count={count})\n")

    # 
    print(f" {name} train accuracy overall: {np.mean(train_acc_list):.3f} ± {np.std(train_acc_list):.3f}")
    print(f" {name} test  accuracy overall: {np.mean(acc_list):.3f} ± {np.std(acc_list):.3f}")
    print(f" {name}          f1 overall: {np.mean(f1_list):.3f} ± {np.std(f1_list):.3f}\n")

    #the variables we save for creating plots
    accuracy_scores[name] = acc_list
    f1_scores[name]       = f1_list
    training_accuracy_scores[name] = train_acc_list   



In [ ]:
# Fold-wise Runtime comparison
plt.figure(figsize=(8, 5))
for name, t_list in runtime_scores.items():
    xs = np.arange(1, len(t_list) + 1)
    line, = plt.plot(xs, t_list, marker='o', label=name)
    for x, t in zip(xs, t_list):
        plt.annotate(f"{t:.2f}s", xy=(x, t),
                     xytext=(0, 5), textcoords="offset points",
                     ha='center', va='bottom', fontsize=8)
    mean_t = np.mean(t_list)
    plt.hlines(mean_t, xs[0], xs[-1],
               linestyles='--', colors=line.get_color(),
               label=f"{name} mean {mean_t:.2f}s")
plt.xticks(xs)
plt.xlabel('Fold')
plt.ylabel('Runtime (s)')
plt.title('Fold-wise Runtime Comparison for Models')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#performance vs time
accuracy_mean = { name: np.mean(vals) for name, vals in accuracy_scores.items() }
accuracy_std  = { name: np.std(vals)  for name, vals in accuracy_scores.items() }
runtime_mean  = { name: np.mean(vals) for name, vals in runtime_scores.items() }
runtime_std   = { name: np.std(vals)  for name, vals in runtime_scores.items() }

models     = list(accuracy_mean.keys())
acc_means  = [accuracy_mean[m] for m in models]
acc_stds   = [accuracy_std[m]  for m in models]
time_means = [runtime_mean[m]  for m in models]
time_stds  = [runtime_std[m]   for m in models]

# error scatter 
plt.figure(figsize=(6,6))
plt.errorbar(
    time_means, acc_means,
    xerr=time_stds, yerr=acc_stds,
    fmt='o', capsize=5
)

# coordinate
for m, x, y in zip(models, time_means, acc_means):
    plt.annotate(
        f"{m} ({x:.2f}s, {y*100:.2f}%)",
        xy=(x, y),
        xytext=(5, -5),
        textcoords="offset points",
        ha='left',
        va='top',
        fontsize=8
    )
plt.xlabel('Avg. Runtime (s)')
plt.ylabel('Avg. Accuracy')
plt.title('Model Performance vs. Runtime')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
for name in pipelines.keys():
    xs = np.arange(1, len(training_accuracy_scores[name]) + 1)
    plt.figure(figsize=(6, 4))

    # train set accuracy
    train_line, = plt.plot(
        xs,
        training_accuracy_scores[name],
        marker='o', linestyle='-',
        label=f"{name} Train"
    )
    for x, y in zip(xs, training_accuracy_scores[name]):
        plt.annotate(
            f"{y:.3f}", xy=(x, y), xytext=(0, 5),
            textcoords="offset points", ha='center', va='bottom', fontsize=8
        )

    # validation(test) set accuracy
    val_line, = plt.plot(
        xs,
        accuracy_scores[name],
        marker='s', linestyle='-',
        label=f"{name} Validation"
    )
    for x, y in zip(xs, accuracy_scores[name]):
        plt.annotate(
            f"{y:.3f}", xy=(x, y), xytext=(0, -10),
            textcoords="offset points", ha='center', va='top', fontsize=8
        )

    # average line
    mean_train = np.mean(training_accuracy_scores[name])
    plt.hlines(
        mean_train, xs[0], xs[-1],
        linestyles='--', colors=train_line.get_color(),
        label=f"{name} Train mean {mean_train:.3f}"
    )

    mean_val = np.mean(accuracy_scores[name])
    plt.hlines(
        mean_val, xs[0], xs[-1],
        linestyles='--', colors=val_line.get_color(),
        label=f"{name} Validation mean {mean_val:.3f}"
    )

    plt.xticks(xs)
    plt.xlabel('Fold')
    plt.ylabel('Accuracy')
    plt.title(f'{name}: Training vs Validation Accuracy with Mean Lines')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# per-fold Macro-F1 similarly
plt.figure()
for name, sc_list in f1_scores.items():
    xs = np.arange(1, len(sc_list)+1)
    plt.plot(xs, sc_list, marker='o', label=name)
    for x, y in zip(xs, sc_list):
        plt.annotate(f"{y:.3f}", xy=(x,y), xytext=(0,10),
                     textcoords="offset points", ha='center', va='bottom', fontsize=8)
    mean_f1 = np.mean(sc_list)
    plt.hlines(mean_f1, xs[0], xs[-1], linestyles='--',
               label=f"{name} mean {mean_f1:.3f}")

plt.xticks(xs)
plt.xlabel('Fold')
plt.ylabel('Macro-F1')
plt.title('Fold-wise Macro-F1 & Mean for 3 Models')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()